In [ ]:
# Configure · Add the validation dataset and Gemma 4. Select GPU T4 ×2 and enable Internet.
INPUT_DIR = ""  # Optional root folder of the extracted validation dataset.
CANDIDATES_PATH = ""  # Optional path to newer generation output.
MODEL_PATH = ""  # Optional path to the attached Gemma GGUF.
BATCH_SIZE = 2
JOURNEYS = 5  # Use None to validate all candidate journeys.
OUTPUT_DIR = "/kaggle/working/output"

from pathlib import Path
from runpy import run_path
from tempfile import TemporaryDirectory

root = Path(INPUT_DIR or "/kaggle/input")
scripts = [path for path in root.rglob("src/validate.py") if (path.parents[1] / "input/blueprints.jsonl").is_file()]
if len(scripts) != 1:
    raise ValueError("Attach one Stage 2 validation dataset or set INPUT_DIR.")
runner = run_path(str(scripts[0]))
environment = TemporaryDirectory(prefix="stage2-deepeval-")
validation_python = runner["prepare_environment"](environment.name)


In [ ]:
# Load · Set the paths and journey count for validation.
args = ["--batch-size", str(BATCH_SIZE), "--output-dir", str(OUTPUT_DIR)]
args += ["--all"] if JOURNEYS is None else ["--journeys", str(JOURNEYS)]
if CANDIDATES_PATH:
    args += ["--candidates", str(CANDIDATES_PATH)]
if MODEL_PATH:
    args += ["--model", str(MODEL_PATH)]


In [ ]:
# Validate · Run the DeepEval judge with the checked environment.
runner["run_validator"](validation_python, args)
